# This code splits the existing clean data into a 70-15-15 train-test-val split.

## Mounting Drive and Importing Libraries

In [1]:
# mounting drive
from google.colab import drive
drive.mount('/content/drive')

# importing libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import os
from sklearn.model_selection import GroupShuffleSplit

Mounted at /content/drive


# Data Loading

In [2]:
# loading the data
path = "/content/drive/MyDrive/Colab Notebooks/SCC 450 - Group Project/Week 10/Data/DATA_CLEAN.csv"
df = pd.read_csv(path, encoding="latin1")
df.head()

/tmp/ipython-input-3565417973.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, encoding="latin1")


,SITECODE,LCODE,SAMPLE_DATE,FIELDNAME,VALUE,FROMDATE,SPERIOD_D,YEAR,WEEK_NEW,98_PERCENTILE,...,SDATE,DRYTMP,NETRAD,SOLAR,STMP10,STMP30,SURWET,WDIR,WSPEED,RAIN
0,T01,1,1993-01-03,XX,0,1993-01-02,1,1993,1,0.0,...,1993-01-02,-2.345833,4.583333,10.500000,0.429167,2.958333,5.875000,0.000000,0.000000,0.0
1,T08,1,1993-01-03,XX,0,1993-01-02,1,1993,1,0.0,...,1993-01-02,-2.733333,3.333333,12.000000,4.155075,4.375149,39.838371,143.916667,0.970833,0.0
2,T08,1,1993-01-04,XX,0,1993-01-03,1,1993,1,0.0,...,1993-01-03,-2.950000,13.083333,50.666667,4.155075,4.375149,39.838371,141.375000,2.775000,0.2
3,T01,1,1993-01-04,XX,0,1993-01-03,1,1993,1,0.0,...,1993-01-03,-1.541667,17.708333,45.458333,0.154167,2.854167,20.666667,77.250000,1.000000,0.0
4,T08,1,1993-01-05,XX,0,1993-01-04,1,1993,1,0.0,...,1993-01-04,2.283333,3.791667,12.000000,4.155075,4.375149,39.838371,188.041667,2.975000,1.6


# Splitting the Data

Performing two splits: one species level, and one site level

Result: two different versions, one will be for Scientist 1 and the other for Scientist 2 just at least for rn while we tackle modelling separately

In [3]:
# bc some FIELDNAMEs are XX n some are numbers
df["FIELDNAME"] = df["FIELDNAME"].astype(str)

# some FIELDNAMEs only have 1 count
# so impossible to split across the train-test-val set fairly
counts = df["FIELDNAME"].value_counts()
major_species = counts[counts >= 20].index

# making a class "OTHER" for species w less than 20 counts
df["SPECIES_BIN"] = df["FIELDNAME"].where(df["FIELDNAME"].isin(major_species), "OTHER")

# dataset: species-level
train_A, temp_A = train_test_split(
    df,
    test_size=0.30,
    # first gonna preserve species data
    stratify=df["SPECIES_BIN"],
    random_state=41
)

val_A, test_A = train_test_split(
    temp_A,
    test_size=0.5,
    stratify=temp_A["SPECIES_BIN"],
    random_state=41
)

# printing length of train, validation, and test datasets
print(len(train_A), len(val_A), len(test_A))


173028 37077 37078


In [4]:
# saving the split for species level
save_path = "/content/drive/MyDrive/Colab Notebooks/SCC 450 - Group Project/Week 10/Data/"
os.makedirs(save_path, exist_ok=True)

train_A.to_csv(f"{save_path}/train_species.csv", encoding="latin1", index=False)
val_A.to_csv(f"{save_path}/val_species.csv", encoding="latin1", index=False)
test_A.to_csv(f"{save_path}/test_species.csv", encoding="latin1", index=False)



In [5]:
# checking
path = "/content/drive/MyDrive/Colab Notebooks/SCC 450 - Group Project/Week 10/Data/"
for root, dirs, files in os.walk(path):
    print(root)
    for f in files:
        print("  ", f)


/content/drive/MyDrive/Colab Notebooks/SCC 450 - Group Project/Week 10/Data/
   Moth Data Merged.csv
   Pivoted Weather Data.csv
   Primary Moth Data CLEAN.csv
   Secondary Moth Data CLEAN(traps).csv
   DATA_CLEAN(no_moth_duplicates).csv
   DATA_CLEAN.csv
   train_species.csv
   val_species.csv
   test_species.csv
